In [ ]:
import os
import numpy as np
from qdrant_client import QdrantClient
from sklearn.metrics.pairwise import rbf_kernel, cosine_similarity
from dotenv import load_dotenv
import plotly.express as px

load_dotenv(override=True)
QDRANT_URL = "http://qdrant:6333"
QDRANT_API_KEY = os.environ["QDRANT_API_KEY"]
client = QdrantClient(
        url=QDRANT_URL,
        api_key=QDRANT_API_KEY,
        timeout=30
    )

def calculate_immd(X, Y, gammas=[1e-3, 1e-2, 0.1, 1, 10]):
    """
    Calcule l'Integrated Maximum Mean Discrepancy (IMMD) 
    en moyennant le MMD sur plusieurs échelles de gamma.
    """
    mmd_scales = []
    for g in gammas:
        xx = rbf_kernel(X, X, gamma=g)
        yy = rbf_kernel(Y, Y, gamma=g)
        xy = rbf_kernel(X, Y, gamma=g)
        mmd = np.mean(xx) + np.mean(yy) - 2 * np.mean(xy)
        mmd_scales.append(mmd)
    
    return np.mean(mmd_scales)

def get_data_for_code(client, collection, code):
    # Récupération par filtre sur le payload
    res = client.scroll(
        collection_name=collection,
        scroll_filter={"must": [{"key": "code", "match": {"value": code}}]},
        limit=500,
        with_vectors=True,
        with_payload=True
    )[0]
    
    if not res:
        return None, None
    
    vecs = np.array([r.vector for r in res])
    labels = [r.payload['label'] for r in res]
    return vecs, labels

def evaluate_metrics(code, orig_col, synth_col):    
    # Extraction
    X_orig, labels_orig = get_data_for_code(client, orig_col, code)
    Y_synth, labels_synth = get_data_for_code(client, synth_col, code)
    
    if X_orig is None or Y_synth is None:
        return "Code non trouvé ou données manquantes."

    # 1. IMMD (Integrated MMD)
    immd_score = calculate_immd(X_orig, Y_synth)

    # 2. Similarité des centroïdes
    c_orig = X_orig.mean(axis=0).reshape(1, -1)
    c_synth = Y_synth.mean(axis=0).reshape(1, -1)
    centroid_sim = cosine_similarity(c_orig, c_synth)[0][0]

    # 3. Similarité maximale entre labels (Max Sim)
    sim_matrix = cosine_similarity(X_orig, Y_synth)
    max_sim = np.max(sim_matrix)

    # 4. DCScore (Data Coverage)
    # Ratio de points ayant au moins un voisin très proche (>0.9 de sim)
    threshold = 0.9
    dc_orig = np.mean(np.max(sim_matrix, axis=1) > threshold)
    dc_synth = np.mean(np.max(sim_matrix, axis=0) > threshold)

    return {
        "code": code,
        "len_orig": len(X_orig),
        "len_synth": len(Y_synth),
        "immd": immd_score,
        "centroid_sim": centroid_sim,
        "max_label_sim": max_sim,
        "dc_score_orig": dc_orig,
        "dc_score_synth": dc_synth
    }

/tmp/ipykernel_21704/2050141161.py:10: UserWarning: Api key is used with an insecure connection.
  client = QdrantClient(


In [3]:
code_hits_real = client.facet(
    collection_name="original_cleaned",
    key="code",
    exact=True,
    limit=1000
).hits

In [45]:
valid_codes = [code_hit.value for code_hit in code_hits_real if code_hit.count >= 50]

In [49]:
from tqdm import tqdm

metrics = {}
for code in tqdm(valid_codes):
    metrics[code] = evaluate_metrics(code, "original_cleaned", "naive_synth")

100%|██████████| 373/373 [09:00<00:00,  1.45s/it]


In [23]:
len(code_hits_real)

747

In [4]:
import pandas as pd

freq_real = {"count":{code_hit.value: code_hit.count for code_hit in code_hits_real}}
df_freq_real = pd.DataFrame(freq_real).reset_index(names="code")

In [55]:
df_metrics = pd.DataFrame(metrics).transpose().reset_index(drop=True)

In [61]:
df_metrics

,code,len_orig,len_synth,immd,centroid_sim,max_label_sim,dc_score_orig,dc_score_synth
0,68.20H,500,100,0.036566,0.87041,0.926379,0.02,0.03
1,70.20Y,500,50,0.037959,0.924665,0.944656,0.204,0.02
2,68.20G,500,50,0.073033,0.920748,0.98392,0.732,0.26
3,96.22Y,500,50,0.026004,0.93557,0.990181,0.116,0.12
4,62.10Y,500,150,0.022864,0.932297,0.949994,0.042,0.066667
...,...,...,...,...,...,...,...,...
368,46.15Y,52,50,0.020617,0.958852,0.922909,0.019231,0.02
369,42.91Y,50,50,0.040576,0.886767,0.913403,0.12,0.06
370,47.53Y,50,50,0.025603,0.945145,0.870452,0.0,0.0
371,49.41H,50,150,0.031768,0.923445,0.944631,0.14,0.04


In [ ]:
fig = px.scatter(df_metrics, x="dc_score_orig", y="dc_score_synth")
fig.show()

In [ ]:
fig = px.bar(df_freq_real, x="code", y="count", log_y=True)
fig.add_vline(x=747/2, annotation_text="Half of the codes", line_dash="dash")
fig.add_hline(y=50, line_color="red")

In [50]:
df_freq_real

,code,count
0,68.20H,71399
1,70.20Y,63150
2,68.20G,27741
3,96.22Y,22422
4,62.10Y,20717
...,...,...
742,91.41Y,1
743,91.42Y,1
744,97.00Y,1
745,98.10Y,1


In [7]:
code_hits_synth = client.facet(
    collection_name="naive_synth",
    key="code",
    exact=True,
    limit=1000
).hits

In [8]:
freq_synth = {"count":{code_hit.value: code_hit.count for code_hit in code_hits_synth}}
df_freq_synth = pd.DataFrame(freq_synth).reset_index(names="code")

In [16]:
px.bar(df_freq_synth, x="code", y="count", log_y=True)